In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D3 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D3 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import math
import re
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D3"
BRANCH_ID = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_REFERENCE_COUNT = 70
REFERENCE_PERIOD = "May 2024"

EXPECTED_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Value",
    "Unit",
    "Reference Period"
]

REFERENCE_FIELDS = EXPECTED_FIELDS + [
    "Source Location"
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Reference Period"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit"
]

ALLOWED_UNITS = {
    "workers",
    "million workers",
    "percent",
    "USD"
}

NUMERIC_TOLERANCE = 1e-9

FALLBACK_MIN_OCCUPATION_SIMILARITY = 0.75
FALLBACK_MIN_TOTAL_SCORE = 0.72

OUTPUT_DIR = Path(
    "outputs_D3_validation_branch_B"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Validation configured.")
print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID, "-", BRANCH_NAME)

In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D3_reference_values.csv
#   2) D3_branch_B_parsed_extraction.json
#   3) D3_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    f for f in uploaded_files
    if f.lower().endswith(".csv")
]

json_files = [
    f for f in uploaded_files
    if f.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one Stage 1 reference-values CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]

PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:

    with open(
        file_name,
        "r",
        encoding="utf-8-sig"
    ) as f:
        obj = json.load(f)

    if (
        isinstance(obj, dict)
        and isinstance(obj.get("records"), list)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "structurally_evaluable" in obj
        and "record_schema_valid" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D3 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D3 Branch B technical diagnostics."
    )

print("Reference values:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)

In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    extraction_json = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)

if extraction_json.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Parsed extraction document_id does not match D3."
    )

if extraction_json.get("branch") != BRANCH_ID:
    raise ValueError(
        "Parsed extraction branch does not match Branch B."
    )

if technical_diagnostics.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Technical diagnostics document_id does not match D3."
    )

if technical_diagnostics.get("branch") != BRANCH_ID:
    raise ValueError(
        "Technical diagnostics branch does not match Branch B."
    )

df_ref = pd.read_csv(
    REFERENCE_FILE,
    encoding="utf-8-sig"
)

df_ext_raw = pd.DataFrame(
    extraction_json["records"]
)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8192),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


input_provenance = {
    "reference_file":
        REFERENCE_FILE,

    "reference_sha256":
        sha256_file(REFERENCE_FILE),

    "parsed_extraction_file":
        PARSED_EXTRACTION_FILE,

    "parsed_extraction_sha256":
        sha256_file(PARSED_EXTRACTION_FILE),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,

    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE)
}

print("Reference shape:", df_ref.shape)
print("Extraction shape:", df_ext_raw.shape)

In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(
    structurally_evaluable
)

schema_diagnostics = {
    "valid_json":
        bool(
            technical_diagnostics.get(
                "valid_json",
                False
            )
        ),

    "record_schema_valid":
        bool(
            technical_diagnostics.get(
                "record_schema_valid",
                False
            )
        ),

    "field_types_valid":
        bool(
            technical_diagnostics.get(
                "field_types_valid",
                False
            )
        ),

    "structurally_evaluable":
        structurally_evaluable,

    "schema_validity":
        schema_validity
}

print("Imported Branch B technical/schema status:")
print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 5. Verify Stage 1 reference and extraction fields
# ============================================================

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = (
        text
        .replace("\u00a0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def normalize_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()
    if not text:
        return np.nan

    text = (
        text
        .replace("$", "")
        .replace("%", "")
        .replace(",", "")
        .replace("\u00a0", "")
        .replace(" ", "")
    )

    if text.startswith("(") and text.endswith(")"):
        text = "-" + text[1:-1]

    try:
        return float(text)
    except (TypeError, ValueError):
        return np.nan


def numbers_match(reference_value, extracted_value):
    ref_num = normalize_number(reference_value)
    ext_num = normalize_number(extracted_value)

    if pd.isna(ref_num) and pd.isna(ext_num):
        return True

    if pd.isna(ref_num) or pd.isna(ext_num):
        return False

    return math.isclose(
        ref_num,
        ext_num,
        rel_tol=0.0,
        abs_tol=NUMERIC_TOLERANCE
    )


def text_similarity(left, right):
    left = normalize_text(left)
    right = normalize_text(right)

    if not left and not right:
        return 1.0
    if not left or not right:
        return 0.0
    if left == right:
        return 1.0

    sequence_score = SequenceMatcher(None, left, right).ratio()

    left_tokens = set(re.findall(r"[a-z0-9]+", left))
    right_tokens = set(re.findall(r"[a-z0-9]+", right))

    if left_tokens or right_tokens:
        token_score = (
            len(left_tokens & right_tokens)
            / len(left_tokens | right_tokens)
        )
    else:
        token_score = 0.0

    return max(sequence_score, token_score)


def occupation_similarity(left, right):
    left_norm = normalize_text(left)
    right_norm = normalize_text(right)

    score = text_similarity(left_norm, right_norm)

    if (
        left_norm
        and right_norm
        and (
            left_norm in right_norm
            or right_norm in left_norm
        )
    ):
        score = max(score, 0.95)

    return score


In [ ]:
# ============================================================
# 6. Verify uniqueness of the fixed Stage 1 identity
# ============================================================

def strict_identity_key(row):
    return tuple(
        normalize_text(row[field])
        for field in ALIGNMENT_IDENTITY_FIELDS
    )


df_ref["_strict_key"] = df_ref.apply(
    strict_identity_key,
    axis=1
)

df_ext["_strict_key"] = df_ext.apply(
    strict_identity_key,
    axis=1
)

reference_duplicate_count = int(
    df_ref["_strict_key"]
    .duplicated(keep=False)
    .sum()
)

if reference_duplicate_count > 0:
    raise ValueError(
        "The fixed Stage 1 reference identity is not unique."
    )

print(
    "Reference duplicate identity rows:",
    reference_duplicate_count
)

print(
    "Extraction duplicate identity rows:",
    int(
        df_ext["_strict_key"]
        .duplicated(keep=False)
        .sum()
    )
)

used_extraction = set()
matched_pairs = []

extraction_key_index = {}
for ext_idx, key in df_ext["_strict_key"].items():
    extraction_key_index.setdefault(key, []).append(ext_idx)

for ref_idx, ref_row in df_ref.iterrows():
    candidates = [
        idx
        for idx in extraction_key_index.get(ref_row["_strict_key"], [])
        if idx not in used_extraction
    ]

    if candidates:
        ext_idx = candidates[0]
        used_extraction.add(ext_idx)
        matched_pairs.append({
            "reference_index": ref_idx,
            "extraction_index": ext_idx,
            "match_method": "strict_identity",
            "alignment_score": 1.0
        })

matched_reference = {
    pair["reference_index"]
    for pair in matched_pairs
}

candidate_pairs = []

for ref_idx, ref_row in df_ref.iterrows():
    if ref_idx in matched_reference:
        continue

    for ext_idx, ext_row in df_ext.iterrows():
        if ext_idx in used_extraction:
            continue

        if (
            normalize_text(ref_row["Reference Period"])
            != normalize_text(ext_row["Reference Period"])
        ):
            continue

        section_score = text_similarity(
            ref_row["Section"],
            ext_row["Section"]
        )
        indicator_score = text_similarity(
            ref_row["Indicator"],
            ext_row["Indicator"]
        )
        occupation_score = occupation_similarity(
            ref_row["Occupation or Group"],
            ext_row["Occupation or Group"]
        )

        total_score = (
            0.20 * section_score
            + 0.25 * indicator_score
            + 0.55 * occupation_score
        )

        if (
            occupation_score >= FALLBACK_MIN_OCCUPATION_SIMILARITY
            and total_score >= FALLBACK_MIN_TOTAL_SCORE
        ):
            candidate_pairs.append({
                "reference_index": ref_idx,
                "extraction_index": ext_idx,
                "alignment_score": total_score,
                "section_score": section_score,
                "indicator_score": indicator_score,
                "occupation_score": occupation_score
            })

candidate_pairs.sort(
    key=lambda x: (
        -x["alignment_score"],
        -x["occupation_score"],
        x["reference_index"],
        x["extraction_index"]
    )
)

for candidate in candidate_pairs:
    ref_idx = candidate["reference_index"]
    ext_idx = candidate["extraction_index"]

    if ref_idx in matched_reference or ext_idx in used_extraction:
        continue

    matched_reference.add(ref_idx)
    used_extraction.add(ext_idx)

    matched_pairs.append({
        **candidate,
        "match_method": "descriptive_fallback"
    })

matched_pairs = sorted(
    matched_pairs,
    key=lambda x: x["reference_index"]
)

matched_reference_indices = {
    pair["reference_index"]
    for pair in matched_pairs
}

missing_reference_indices = [
    idx for idx in df_ref.index
    if idx not in matched_reference_indices
]

unsupported_extraction_indices = [
    idx for idx in df_ext.index
    if idx not in used_extraction
]

print("Aligned records:", len(matched_pairs))
print("Missing expected records:", len(missing_reference_indices))
print("Unsupported extracted records:", len(unsupported_extraction_indices))
print(
    pd.Series(
        [pair["match_method"] for pair in matched_pairs],
        dtype="object"
    ).value_counts()
)


In [ ]:
# ============================================================
# 7. Compare fields after alignment
# ============================================================

record_rows = []
field_rows = []

for pair in matched_pairs:

    ref = df_ref.loc[
        pair["reference_index"]
    ]

    ext = df_ext.loc[
        pair["extraction_index"]
    ]

    field_matches = {
        "Section":
            normalize_text(ref["Section"])
            == normalize_text(ext["Section"]),

        "Indicator":
            normalize_text(ref["Indicator"])
            == normalize_text(ext["Indicator"]),

        "Occupation or Group":
            normalize_text(ref["Occupation or Group"])
            == normalize_text(
                ext["Occupation or Group"]
            ),

        "Value":
            numbers_match(
                ref["Value"],
                ext["Value"]
            ),

        "Unit":
            normalize_text(ref["Unit"])
            == normalize_text(ext["Unit"]),

        "Reference Period":
            normalize_text(
                ref["Reference Period"]
            )
            == normalize_text(
                ext["Reference Period"]
            )
    }

    primary_correct = all(
        field_matches[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    identity_fields_match = all(
        field_matches[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    status = (
        "fully_correct"
        if primary_correct
        else "discrepant"
    )

    mismatched_fields = [
        field
        for field, is_match
        in field_matches.items()
        if not is_match
    ]

    record_rows.append({
        "Reference Record ID":
            f"D3-REF-{pair['reference_index'] + 1:03d}",

        "Extraction Record ID":
            f"D3-B-{pair['extraction_index'] + 1:03d}",

        "Section_ref":
            ref["Section"],

        "Section_ext":
            ext["Section"],

        "Indicator_ref":
            ref["Indicator"],

        "Indicator_ext":
            ext["Indicator"],

        "Occupation or Group_ref":
            ref["Occupation or Group"],

        "Occupation or Group_ext":
            ext["Occupation or Group"],

        "Value_ref":
            ref["Value"],

        "Value_ext":
            ext["Value"],

        "Unit_ref":
            ref["Unit"],

        "Unit_ext":
            ext["Unit"],

        "Reference Period_ref":
            ref["Reference Period"],

        "Reference Period_ext":
            ext["Reference Period"],

        "Source Location":
            ref["Source Location"],

        "Match Method":
            pair["match_method"],

        "Alignment Score":
            pair.get("alignment_score"),

        "Section_match":
            field_matches["Section"],

        "Indicator_match":
            field_matches["Indicator"],

        "Occupation or Group_match":
            field_matches["Occupation or Group"],

        "Value_match":
            field_matches["Value"],

        "Unit_match":
            field_matches["Unit"],

        "Reference Period_match":
            field_matches["Reference Period"],

        "primary_correct":
            primary_correct,

        "alignment_identity_fields_match":
            identity_fields_match,

        "alignment_identity_discrepancy":
            not identity_fields_match,

        "record_status":
            status,

        "mismatched_fields":
            ", ".join(mismatched_fields)
    })

    for field in EXPECTED_FIELDS:

        field_rows.append({
            "Reference Record ID":
                f"D3-REF-{pair['reference_index'] + 1:03d}",

            "Extraction Record ID":
                f"D3-B-{pair['extraction_index'] + 1:03d}",

            "Field":
                field,

            "Field Match":
                field_matches[field],

            "Match Method":
                pair["match_method"],

            "Source Location":
                ref["Source Location"]
        })


record_validation_df = pd.DataFrame(
    record_rows
)

field_validation_df = pd.DataFrame(
    field_rows
)

record_validation_df.head()

In [ ]:
# ============================================================
# 8. Create missing, unsupported and discrepant outputs
# ============================================================

missing_records = (
    df_ref.loc[
        missing_reference_indices
    ].copy()
)

unsupported_records = (
    df_ext.loc[
        unsupported_extraction_indices
    ].copy()
)

if "_strict_key" in missing_records.columns:
    missing_records = (
        missing_records.drop(
            columns=["_strict_key"]
        )
    )

if "_strict_key" in unsupported_records.columns:
    unsupported_records = (
        unsupported_records.drop(
            columns=["_strict_key"]
        )
    )

fully_correct_records = (
    record_validation_df[
        record_validation_df[
            "record_status"
        ] == "fully_correct"
    ].copy()
)

discrepant_records = (
    record_validation_df[
        record_validation_df[
            "record_status"
        ] == "discrepant"
    ].copy()
)

alignment_identity_discrepancies = (
    record_validation_df[
        record_validation_df[
            "alignment_identity_discrepancy"
        ]
    ].copy()
)

print(
    "Fully correct:",
    len(fully_correct_records)
)

print(
    "Discrepant:",
    len(discrepant_records)
)

print(
    "Alignment-identity discrepancies:",
    len(alignment_identity_discrepancies)
)

print(
    "Missing:",
    len(missing_records)
)

print(
    "Unsupported/hallucinated:",
    len(unsupported_records)
)

In [ ]:
# ============================================================
# 9. Calculate common validation metrics
# ============================================================

N_REF = int(
    len(df_ref)
)

N_EXT = int(
    len(df_ext)
)

N_ALIGNED = int(
    len(record_validation_df)
)

N_CORRECT = int(
    len(fully_correct_records)
)

N_DISCREPANT = int(
    len(discrepant_records)
)

N_MISSING = int(
    len(missing_records)
)

N_HALLUCINATED = int(
    len(unsupported_records)
)

completeness = (
    N_ALIGNED / N_REF
    if N_REF else 0.0
)

missing_rate = (
    N_MISSING / N_REF
    if N_REF else 0.0
)

record_precision = (
    N_CORRECT / N_EXT
    if N_EXT else 0.0
)

record_recall = (
    N_CORRECT / N_REF
    if N_REF else 0.0
)

record_f1 = (
    2
    * record_precision
    * record_recall
    / (
        record_precision
        + record_recall
    )
    if (
        record_precision
        + record_recall
    )
    else 0.0
)

hallucination_rate = (
    N_HALLUCINATED / N_EXT
    if N_EXT else 0.0
)

discrepancy_rate = (
    N_DISCREPANT / N_ALIGNED
    if N_ALIGNED else 0.0
)

field_accuracy_among_aligned = {}

for field in EXPECTED_FIELDS:

    rows = field_validation_df[
        field_validation_df[
            "Field"
        ] == field
    ]

    field_accuracy_among_aligned[
        field
    ] = (
        float(
            rows["Field Match"].mean()
        )
        if len(rows)
        else 0.0
    )


primary_field_validation_df = (
    field_validation_df[
        field_validation_df[
            "Field"
        ].isin(
            PRIMARY_CORRECTNESS_FIELDS
        )
    ]
)

correct_field_instances = int(
    primary_field_validation_df[
        "Field Match"
    ].sum()
)

expected_field_instances = int(
    N_REF
    * len(
        PRIMARY_CORRECTNESS_FIELDS
    )
)

field_accuracy = (
    correct_field_instances
    / expected_field_instances
    if expected_field_instances
    else 0.0
)

print(
    "Common validation metrics calculated."
)

In [ ]:
# ============================================================
# 10. Field-level error summary
# ============================================================

field_error_summary = []

for field in EXPECTED_FIELDS:

    rows = field_validation_df[
        field_validation_df[
            "Field"
        ] == field
    ]

    correct_aligned = int(
        rows["Field Match"].sum()
    )

    incorrect_aligned = int(
        len(rows)
        - correct_aligned
    )

    field_error_summary.append({
        "field":
            field,

        "used_in_alignment_identity":
            field in ALIGNMENT_IDENTITY_FIELDS,

        "used_in_primary_correctness":
            field in PRIMARY_CORRECTNESS_FIELDS,

        "aligned_records_evaluated":
            int(len(rows)),

        "correct_values_among_aligned":
            correct_aligned,

        "incorrect_values_among_aligned":
            incorrect_aligned,

        "accuracy_among_aligned":
            (
                round(
                    correct_aligned
                    / len(rows),
                    4
                )
                if len(rows)
                else 0.0
            ),

        "missing_expected_instances":
            N_MISSING,

        "overall_correct_instances":
            correct_aligned,

        "overall_expected_instances":
            N_REF,

        "overall_accuracy_against_reference":
            (
                round(
                    correct_aligned
                    / N_REF,
                    4
                )
                if N_REF
                else 0.0
            )
    })


field_error_summary_df = (
    pd.DataFrame(
        field_error_summary
    )
)

field_error_summary_df

In [ ]:
# ============================================================
# 11. Section-level performance
# ============================================================

section_rows = []

for section, ref_group in df_ref.groupby(
    "Section",
    dropna=False
):

    ref_indices = set(
        ref_group.index
    )

    aligned_section = (
        record_validation_df[
            record_validation_df[
                "Reference Record ID"
            ].apply(
                lambda x:
                    int(
                        str(x).split("-")[-1]
                    ) - 1
                    in ref_indices
            )
        ]
    )

    section_rows.append({
        "Section":
            section,

        "Reference Records":
            len(ref_group),

        "Aligned Records":
            len(aligned_section),

        "Fully Correct Records":
            int(
                (
                    aligned_section[
                        "record_status"
                    ]
                    == "fully_correct"
                ).sum()
            ),

        "Discrepant Records":
            int(
                (
                    aligned_section[
                        "record_status"
                    ]
                    == "discrepant"
                ).sum()
            ),

        "Alignment Identity Discrepancies":
            int(
                aligned_section[
                    "alignment_identity_discrepancy"
                ].sum()
            ),

        "Missing Records":
            (
                len(ref_group)
                - len(aligned_section)
            )
    })


section_performance_df = pd.DataFrame(
    section_rows
)

section_performance_df

In [ ]:
# ============================================================
# 12. Build final Branch B validation summary
# ============================================================

summary = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "branch_name":
        BRANCH_NAME,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "hallucinated_records":
        N_HALLUCINATED,

    "completeness":
        round(
            completeness,
            4
        ),

    "missing_rate":
        round(
            missing_rate,
            4
        ),

    "record_precision_exact":
        round(
            record_precision,
            4
        ),

    "record_recall_exact":
        round(
            record_recall,
            4
        ),

    "record_f1_exact":
        round(
            record_f1,
            4
        ),

    "hallucination_rate":
        round(
            hallucination_rate,
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate,
            4
        ),

    "field_accuracy":
        round(
            field_accuracy,
            4
        ),

    "field_accuracy_among_aligned": {
        key:
            round(
                value,
                4
            )
        for key, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "structurally_evaluable":
        structurally_evaluable,

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "comparison_rules_frozen_from_branch_A":
        True,

    "alignment_rules": {
        "strict_identity_first":
            True,

        "controlled_descriptive_fallback":
            True,

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False,

        "fallback_min_occupation_similarity":
            FALLBACK_MIN_OCCUPATION_SIMILARITY,

        "fallback_min_total_score":
            FALLBACK_MIN_TOTAL_SCORE
    },

    "comparison_rules": {
        "text":
            (
                "Unicode NFKC, apostrophe/dash standardisation, "
                "whitespace collapse and case folding"
            ),

        "unit":
            (
                "Text normalisation only; "
                "no semantic unit remapping"
            ),

        "numeric":
            (
                "Direct numerical comparison "
                "in the reported source scale"
            ),

        "numeric_tolerance":
            NUMERIC_TOLERANCE,

        "rounded_million_values_expanded":
            False
    },

    "normalisation_note":
        (
            "Normalisation is applied only to comparison copies. "
            "The preserved Branch B extraction is not modified."
        ),

    "input_provenance":
        input_provenance
}

print(
    "Final validation summary:"
)

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 13. Validation integrity checks
# ============================================================

assert (
    N_ALIGNED
    + N_MISSING
    == N_REF
)

assert (
    N_ALIGNED
    + N_HALLUCINATED
    == N_EXT
)

assert (
    N_CORRECT
    + N_DISCREPANT
    == N_ALIGNED
)

for metric_name, metric_value in {
    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision":
        record_precision,

    "record_recall":
        record_recall,

    "record_f1":
        record_f1,

    "hallucination_rate":
        hallucination_rate,

    "discrepancy_rate":
        discrepancy_rate,

    "field_accuracy":
        field_accuracy
}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )

print(
    "Validation integrity checks passed."
)

In [ ]:
# ============================================================
# 14. Export validation artefacts
# ============================================================

record_validation_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_validation_detailed.csv",
    index=False
)

field_validation_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_field_validation.csv",
    index=False
)

missing_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_missing_records.csv",
    index=False
)

unsupported_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_hallucinated_records.csv",
    index=False
)

fully_correct_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_fully_correct_records.csv",
    index=False
)

discrepant_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_discrepant_records.csv",
    index=False
)

alignment_identity_discrepancies.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_alignment_identity_discrepancies.csv",
    index=False
)

field_error_summary_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_field_error_summary.csv",
    index=False
)

section_performance_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_B_section_performance.csv",
    index=False
)

with open(
    OUTPUT_DIR
    / "D3_branch_B_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Validation artefacts saved."
)

In [ ]:
# ============================================================
# 15. Download validation artefacts
# ============================================================

for output_file in sorted(
    OUTPUT_DIR.iterdir()
):
    if output_file.is_file():
        files.download(
            output_file
        )